# Assignment 4: LSTM for time-series forecasting (stock price)
Predict the next-day closing price from the previous 60 days using stacked LSTM layers.

In [ ]:
!pip install -q yfinance
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
tf.keras.utils.set_random_seed(42)

## 1. Load data
Downloads AAPL daily prices with yfinance. If the download fails (no internet) a synthetic series is used so the notebook still runs.

In [ ]:
import yfinance as yf
try:
    raw = yf.download("AAPL", start="2015-01-01", end="2025-01-01", progress=False, auto_adjust=True)
    series = raw["Close"].squeeze().dropna()
    assert len(series) > 500
    title = "AAPL close"
except Exception as e:
    print("Download failed, using synthetic data:", e)
    idx = pd.bdate_range("2015-01-01", periods=2500)
    t = np.arange(2500)
    series = pd.Series(100 + 0.05*t + 10*np.sin(t/30) + np.random.randn(2500).cumsum()*0.5, index=idx)
    title = "Synthetic series"
print(series.shape); series.plot(figsize=(10, 3), title=title); plt.show()

## 2. Scale and create sequences
Scaler is fit only on the training part to avoid leakage.

In [ ]:
WINDOW = 60
values = series.values.reshape(-1, 1)
split = int(len(values) * 0.8)

scaler = MinMaxScaler().fit(values[:split])
scaled = scaler.transform(values)

def make_seq(data, start, end):
    X, y = [], []
    for i in range(max(start, WINDOW), end):
        X.append(data[i-WINDOW:i, 0]); y.append(data[i, 0])
    return np.array(X)[..., None], np.array(y)

X_train, y_train = make_seq(scaled, 0, split)
X_test, y_test = make_seq(scaled, split, len(scaled))   # test windows may look back into train: that is fine
print(X_train.shape, X_test.shape)

## 3. LSTM model

In [ ]:
model = keras.Sequential([
    keras.layers.Input((WINDOW, 1)),
    keras.layers.LSTM(64, return_sequences=True),
    keras.layers.Dropout(0.2),
    keras.layers.LSTM(32),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
model.summary()

In [ ]:
cb = [keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
      keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)]
hist = model.fit(X_train, y_train, validation_split=0.1, epochs=60, batch_size=32, callbacks=cb, verbose=1)
plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="val"); plt.legend(); plt.title("Loss"); plt.show()

## 4. Evaluation and forecast plot

In [ ]:
pred = scaler.inverse_transform(model.predict(X_test, verbose=0))
actual = scaler.inverse_transform(y_test.reshape(-1, 1))
rmse = np.sqrt(mean_squared_error(actual, pred)); mae = mean_absolute_error(actual, pred); r2 = r2_score(actual, pred)
mape = np.mean(np.abs((actual - pred) / actual)) * 100
print(f"RMSE={rmse:.3f}  MAE={mae:.3f}  MAPE={mape:.2f}%  R2={r2:.3f}")

dates = series.index[-len(actual):]
plt.figure(figsize=(12, 4))
plt.plot(dates, actual, label="Actual"); plt.plot(dates, pred, label="LSTM prediction")
plt.legend(); plt.title("Test-set forecast"); plt.show()

## 5. Multi-step forecast (next 30 days, recursive)

In [ ]:
window = scaled[-WINDOW:, 0].tolist()
future = []
for _ in range(30):
    nxt = model.predict(np.array(window[-WINDOW:])[None, :, None], verbose=0)[0, 0]
    future.append(nxt); window.append(nxt)
future = scaler.inverse_transform(np.array(future).reshape(-1, 1))
fut_idx = pd.bdate_range(series.index[-1] + pd.Timedelta(days=1), periods=30)
plt.figure(figsize=(12, 4))
plt.plot(series.index[-120:], series.values[-120:], label="History")
plt.plot(fut_idx, future, label="30-day forecast")
plt.legend(); plt.show()

## Conclusion
The stacked LSTM tracks the trend well on one-step-ahead prediction. Recursive multi-step forecasts drift because errors compound, which is expected for financial series. This is an educational model, not trading advice.